# PyTorch Environment Check (Fixed Version)
**Must run in order starting from Cell 1 — do not skip cells.**

In [1]:
# Cell 1: must run first — set environment variables to prevent kernel crash
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'  # MPS unsupported op auto fallback to CPU
print('Environment variables set')

Environment variables set


In [2]:
# Cell 2: Basic info
import torch
import platform

print(f'Python:        {platform.python_version()}')
print(f'PyTorch:       {torch.__version__}')
print(f'CUDA available:     {torch.cuda.is_available()}')
print(f'MPS available:      {torch.backends.mps.is_available()}')

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'\nDevice in use:   {device}')

Python:        3.11.15
PyTorch:       2.11.0
CUDA available:     False
MPS available:      True

Device in use:   mps


In [3]:
# Cell 3: Tensor ops + device transfer
a = torch.randn(3, 3)
b = torch.randn(3, 3)

a_dev = a.to(device)
b_dev = b.to(device)
c_dev = torch.matmul(a_dev, b_dev)

print(f'Tensor a (CPU):\n{a}')
print(f'\na @ b result (computed on {device} ):\n{c_dev}')
print(f'\nresult device: {c_dev.device}')
print('\nTensor ops + device transfer OK')

Tensor a (CPU):
tensor([[ 0.8151, -1.2563, -0.5049],
        [-1.4980,  0.1696,  0.8369],
        [ 0.1651, -0.1207, -0.0087]])

a @ b result (computed on mps ):
tensor([[-0.3251,  3.1488,  1.8262],
        [ 0.5169, -4.2100, -1.3882],
        [-0.0793,  0.3336,  0.2612]], device='mps:0')

result device: mps:0

Tensor ops + device transfer OK


In [4]:
# Cell 4: Autograd
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x + 1   # y = (x+1)², dy/dx = 2x+2
y.backward()

print(f'y = x² + 2x + 1,  x = {x.item():.1f}')
print(f'dy/dx analytic value:    {2*x.item() + 2:.1f}')
print(f'dy/dx autograd: {x.grad.item():.1f}')
assert abs(x.grad.item() - 8.0) < 1e-5
print('\nAutograd OK')

y = x² + 2x + 1,  x = 3.0
dy/dx analytic value:    8.0
dy/dx autograd: 8.0

Autograd OK


In [5]:
# Cell 5: full training loop (fit y = 2x + 1)
import torch.nn as nn
torch.manual_seed(42)

X = torch.linspace(-1, 1, 100).unsqueeze(1).to(device)
Y = 2 * X + 1 + 0.05 * torch.randn_like(X)

model = nn.Linear(1, 1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

for epoch in range(500):
    pred = model(X)
    loss = loss_fn(pred, Y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

w = model.weight.item()
b = model.bias.item()
print(f'Fit result: y = {w:.4f}x + {b:.4f}')
print(f'True params: y = 2.0000x + 1.0000')
print(f'Final Loss: {loss.item():.6f}')
assert abs(w - 2.0) < 0.05 and abs(b - 1.0) < 0.05
print(f'\nNeural network training OK (device: {device})')

Fit result: y = 2.0031x + 1.0079
True params: y = 2.0000x + 1.0000
Final Loss: 0.002217

Neural network training OK (device: mps)


In [6]:
# Cell 6: CPU vs MPS/GPU speed comparison
import time

size = 2000
a = torch.randn(size, size)
b = torch.randn(size, size)

# CPU
t0 = time.time()
for _ in range(10):
    torch.matmul(a, b)
cpu_time = (time.time() - t0) / 10 * 1000

# GPU/MPS
a_dev = a.to(device)
b_dev = b.to(device)
torch.matmul(a_dev, b_dev)  # warm-up

t0 = time.time()
for _ in range(10):
    torch.matmul(a_dev, b_dev)
gpu_time = (time.time() - t0) / 10 * 1000

print(f'Matrix multiply {size}x{size}, averaged over 10  runs')
print(f'CPU  time: {cpu_time:.2f} ms')
print(f'{str(device).upper()} time: {gpu_time:.2f} ms')
if device.type != "cpu":
    print(f'Speedup: {cpu_time/gpu_time:.1f}x')
print('\nSpeed test done')

Matrix multiply 2000x2000, averaged over 10  runs
CPU  time: 11.03 ms
MPS time: 0.03 ms
Speedup: 350.2x

Speed test done


In [7]:
# Cell 7: Summary
print('========== Environment Check Summary ==========')
print(f'  PyTorch version: {torch.__version__}')
print(f'  Compute device:     {device}')
print(f'  Tensor ops:  ')
print(f'  Autograd:     ')
print(f'  Model training:     ')
print(f'  GPU accel:     {"" if device.type != "cpu" else "No GPU detected, using CPU"}')
print('\nAll checks passed. Ready to start learning PyTorch!')

========== Environment Check Summary ==========
  PyTorch version: 2.11.0
  Compute device:     mps
  Tensor ops:  
  Autograd:     
  Model training:     
  GPU accel:     

All checks passed. Ready to start learning PyTorch!


In [8]:
import torchvision
import torchvision.transforms as transforms
from torchvision import models

print(f'torchvision version: {torchvision.__version__}')

# 1. Test transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
print('transforms OK')

# 2. Load pretrained model architecture (without downloading weights)
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

resnet = models.resnet18(weights=None).to(device)
resnet.eval()
print('ResNet18 architecture loaded OK')

# 3. random tensor to simulate image inference
dummy_image = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    output = resnet(dummy_image)
print(f'Inference OK, output shape: {output.shape}')  # should be [1, 1000]

print(f'\ntorchvision fully verified (device: {device})')

torchvision version: 0.26.0
transforms OK
ResNet18 architecture loaded OK
Inference OK, output shape: torch.Size([1, 1000])

torchvision fully verified (device: mps)
